# Anima 2B SnapRef-style training
Separate reference rank-128 and target rank-64 LoRAs on frozen Anima base v1.0. Reference caption defaults to empty. Reuses the saved artist/date index and velocity L2 only. Both LoRA branches and the cs (character + style) reference-mode embedding start from the published BF16 SnapRef v1a adapter, then are finetuned with velocity L2.


In [ ]:
import io, json, os, shutil, stat, subprocess, sys, zipfile
from pathlib import Path
if sys.platform != 'linux' or not Path('/content').is_dir():
    raise RuntimeError('This notebook is for Google Colab only; do not run it locally.')
from google.colab import files, userdata
print('Colab guard passed. No installer has run yet.')


In [ ]:
# Default: clone or fast-forward the committed fork. Upload/existing remain optional.
SOURCE = 'git'  # 'git', 'upload', or 'existing'
REPO_URL = 'https://github.com/TheDucker1/sd-scripts'
BRANCH = 'style_loss'
REPO = Path('/content/sd-scripts')
if SOURCE == 'upload':
    uploaded = files.upload()
    archives = [data for name, data in uploaded.items() if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise RuntimeError('Upload exactly one source-only repository ZIP.')
    destination = Path('/content/moss-upload')
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(archives[0])) as archive:
        if sum(info.file_size for info in archive.infolist()) > 512 * 1024**2:
            raise RuntimeError('Source ZIP exceeds 512 MiB; exclude models, datasets and environments.')
        for info in archive.infolist():
            target = (destination / info.filename).resolve()
            if destination not in target.parents or stat.S_ISLNK(info.external_attr >> 16):
                raise RuntimeError('Unsafe archive entry: ' + info.filename)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as source, target.open('wb') as output:
                    shutil.copyfileobj(source, output)
    candidates = list(destination.rglob('anima_train_style.py'))
    if len(candidates) != 1:
        raise RuntimeError('Expected exactly one current Anima style trainer in the ZIP.')
    REPO = candidates[0].parent
    del archives, uploaded
elif SOURCE == 'git':
    if not REPO_URL:
        raise RuntimeError('Set REPO_URL.')
    if not REPO.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
    else:
        def git_output(*args):
            return subprocess.check_output(['git', '-C', str(REPO), *args], text=True).strip()
        if git_output('remote', 'get-url', 'origin').rstrip('/').removesuffix('.git') != REPO_URL.rstrip('/').removesuffix('.git'):
            raise RuntimeError('Existing checkout has a different origin.')
        if git_output('symbolic-ref', '--short', 'HEAD') != BRANCH:
            raise RuntimeError('Existing checkout is on a different branch.')
        if git_output('status', '--porcelain', '--untracked-files=no'):
            raise RuntimeError('Tracked files have local edits. Commit or stash them before updating; no edits were discarded.')
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO), 'merge', '--ff-only', 'FETCH_HEAD'], check=True)
    print('Source commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
elif SOURCE != 'existing':
    raise ValueError('Unknown SOURCE')
SETUP = REPO / 'tools/colab/setup_moss_style.py'
if not SETUP.is_file():
    raise RuntimeError('Missing setup script; commit and push all new Anima files to the selected fork/branch first.')
print('Using source:', REPO)


In [ ]:
# One-time authentication for this runtime. Never put your token in code or chat.
# Token is inherited by setup/build subprocesses; no persistent HF login.
from getpass import getpass
FORCE_TOKEN_PROMPT = False  # Set True to replace an expired/rejected token.
token = None if FORCE_TOKEN_PROMPT else os.environ.get('HF_TOKEN')
if not token and not FORCE_TOKEN_PROMPT:
    try:
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = None
if not token:
    token = getpass('Hugging Face read token (hidden, this runtime only): ').strip()
if not token or not token.strip():
    raise RuntimeError('Provide a Hugging Face read token for model setup.')
os.environ['HF_TOKEN'] = token.strip()
print('HF token is available for this runtime. No token value is printed or written to disk.')
del token


In [ ]:
DOWNLOAD_MODELS = True
DRIVE_ROOT = Path('/content/drive/MyDrive/moss-training')
DATASET_NAME = 'anima-1000-p10-512'
OLD_RUN_NAME = 'anima-snapref-flow-l2-768-fp32-3'  # Edit if step-074000 is in another Drive folder.
INIT_CHECKPOINT = DRIVE_ROOT / OLD_RUN_NAME / 'step-074000'
TEXT_ENCODER = 'bf16'  # Qwen3-0.6B-Base; no ConvRot profile.
# W4A8/GPTQ, GGUF, and arbitrary replacement encoders are not supported here.
PERSIST_MODELS_TO_DRIVE = False
VENV = Path('/content/moss-venv')
MODELS = Path('/content/anima-models')
if PERSIST_MODELS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    MODELS = Path('/content/drive/MyDrive/anima-models')
print('Environment:', VENV, '\nAssets:', MODELS)


In [ ]:
# This is the only installation/download step. Run it on the Colab VM.
# Setup validates the token before downloading model weights; DINOv3 is no longer needed.
command = [sys.executable, str(SETUP), '--repo', str(REPO), '--repo-url', REPO_URL, '--branch', BRANCH, '--venv', str(VENV), '--models-dir', str(MODELS), '--text-encoder', TEXT_ENCODER]
from google.colab import drive
drive.mount('/content/drive')
for filename in ('adapter_config.json', 'adapter_model.safetensors'):
    if not (INIT_CHECKPOINT / filename).is_file():
        raise FileNotFoundError(INIT_CHECKPOINT / filename)
source_adapter = json.loads((INIT_CHECKPOINT / 'adapter_config.json').read_text())
source_revisions = source_adapter.get('metadata', {}).get('pretrained_revisions', {})
assets = json.loads((REPO / 'configs/anima_colab_models.json').read_text())
assets['models'] = [model for model in assets['models'] if model['name'] not in ('snapref_adapter', 'tagger')]
for model in [*assets['models'], assets['text_encoder_profiles'][TEXT_ENCODER]]:
    revision = source_revisions.get(model['name'], '')
    if len(revision) == 40 and all(character in '0123456789abcdef' for character in revision):
        model['revision'] = revision
asset_manifest = Path('/content/anima-074000-models.json')
asset_manifest.write_text(json.dumps(assets, indent=2) + '\n')
command += ['--manifest', str(asset_manifest)]
if not DOWNLOAD_MODELS:
    command.append('--skip-models')
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
# Inspect metadata only. These cells do not load checkpoint tensors.
print((VENV / 'moss-runtime.json').read_text())
for name in ['resolved_models.json', 'dataset_stream.json']:
    path = MODELS / name
    if path.exists():
        print(name, '->', path)
print('Use this interpreter for later commands:', VENV / 'bin/python')
print('Preparation only: no training has started.')


Setup uses the saved model revisions when available. This run initializes both LoRA branches and the learned mode from step 74,000 and starts at step 1 with a fresh optimizer and schedule.


## Stage the offline artist archive on Colab SSD
Upload the uncompressed image tar from `tools/pack_artist_dataset.py` as `images.tar` inside `MyDrive/moss-training/anima-1000-p10-512/`. It contains the completed manifest, final index and selected images.
The staging cell validates image checksums and builds local tar offsets. Previews read source images directly from this tar. Training reads the cached tags and latents. Staged files survive training process restarts within the same Colab runtime.
Both download selection and training enforce the saved same-artist, distinct-image date window (365 days here). Eligible pair counts are unordered; a roster can contain images more than one year apart, but those images will never be paired with each other.


In [ ]:
# Tiny synthetic offline fixtures only: no dataset fetches, pretrained weights, or GPU.
subprocess.run([str(VENV / 'bin/python'), '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_stage_artist_dataset.py', '-v'], cwd=REPO, check=True)
subprocess.run([str(VENV / 'bin/python'), '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_anima_reference.py', '-v'], cwd=REPO, check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATASET_DRIVE = DRIVE_ROOT / DATASET_NAME
DATA_ARCHIVE = DATASET_DRIVE / 'images.tar'
DATA_STORAGE = 'local'  # Colab SSD. 'ram' uses /dev/shm when enough capacity is available.
RESERVE_RAM_GIB = 12  # RAM left available for model loading, training and CPU prefetch.
if DATA_STORAGE not in ('ram', 'local'):
    raise ValueError("DATA_STORAGE must be 'ram' or 'local'")
INDEX_DIR = Path('/dev/shm' if DATA_STORAGE == 'ram' else '/content') / DATASET_NAME / 'images'
if not DATA_ARCHIVE.is_file():
    raise FileNotFoundError(f'Upload the completed dataset .tar to {DATA_ARCHIVE}, or change DATA_ARCHIVE.')
stage_command = [str(VENV / 'bin/python'), '-u', str(REPO / 'tools/colab/stage_artist_dataset.py'),
                 '--archive', str(DATA_ARCHIVE), '--output', str(INDEX_DIR),
                 '--storage', DATA_STORAGE, '--reserve-ram-gib', str(RESERVE_RAM_GIB)]
subprocess.run(stage_command, cwd=REPO, check=True)
index_manifest = json.loads((INDEX_DIR / 'manifest.json').read_text())
assert index_manifest['complete'] and index_manifest['storage'] == 'local_tar'
print('Ready:', index_manifest['image_count'], 'images;', index_manifest['artist_count'], 'artists;', DATA_STORAGE)
print('Pair window:', index_manifest['pair_max_days'], 'days; no image reads from Drive during training.')


## Stage the precomputed latents and tags
Upload `latents.tar` and `tag-cache.json` alongside `images.tar`.
Run `tools/cache_anima_artist_latents.py` locally with `--vae` (existing weights) or `--download-vae` (downloads only that VAE file). The worker loads only the VAE, encodes at batch size 1, resumes valid NPZs, then packs a separate latent-cache `.tar`. Upload that tar alongside your image tar. Use the same bucket/resize settings and VAE checkpoint as training.
This cell copies the cache archive to the Colab SSD once and unpacks verified NPZs locally. The training process checks dataset contents, preprocessing, dtype and VAE checkpoint hash before reusing it. A mismatch stops with an error instead of silently spending another hour encoding. The VAE is not loaded for encoding when importing a cache; JPEG previews temporarily load it for decoding.
`LATENTS_IN_MEMORY=False` reads only the current latent pair from the local SSD. Set it to True to load CPU tensors into RAM once, with a free-RAM check. This does not put the dataset cache into VRAM. Training batch size is still controlled separately below.


In [ ]:
USE_PRECOMPUTED_LATENTS = True
LATENT_ARCHIVE = DATASET_DRIVE / 'latents.tar'
PRECOMPUTED_LATENT_DIR = Path('/content') / DATASET_NAME / 'latents'
TAG_SOURCE = DATASET_DRIVE / 'tag-cache.json'
TAG_CACHE = PRECOMPUTED_LATENT_DIR.parent / TAG_SOURCE.name
if not TAG_SOURCE.is_file():
    raise FileNotFoundError(f'Upload the offline tag cache to {TAG_SOURCE}.')
LATENTS_IN_MEMORY = False  # Optional CPU RAM cache; otherwise read NPZs from the Colab SSD.
if USE_PRECOMPUTED_LATENTS:
    if not LATENT_ARCHIVE.is_file():
        raise FileNotFoundError(f'Upload the local VAE worker cache tar to {LATENT_ARCHIVE}, or change LATENT_ARCHIVE.')
    subprocess.run([str(VENV / 'bin/python'), '-u', str(REPO / 'tools/colab/stage_anima_latents.py'),
                    '--archive', str(LATENT_ARCHIVE), '--output', str(PRECOMPUTED_LATENT_DIR)], cwd=REPO, check=True)
TAG_CACHE.parent.mkdir(parents=True, exist_ok=True)
temporary_tags = TAG_CACHE.with_suffix('.json.part')
shutil.copyfile(TAG_SOURCE, temporary_tags)
temporary_tags.replace(TAG_CACHE)
print('Offline tags staged:', TAG_CACHE)


## Training settings and persistence
Training uses velocity L2 with a BF16 base and FP32 LoRA weights, gradients and mode embedding. Batch 1 with accumulation 2 gives two pairs per update. Gradient checkpointing starts disabled. If the assigned GPU runs out of memory, set CHECKPOINT_BLOCKS to 28 and rerun the settings and launch cells.

| Setting | Default | Meaning |
|---|---|---|
| PIXEL_BUDGET | 512 × 512 | Maximum area per image for A and B. Aspect buckets match the offline caches. |
| MAX_STEPS | 100000 | Total optimizer updates, including updates completed before resume. |
| RESUME_DATA_MODE | reseed | Draws a fresh dataset seed when resuming this run. |
| PREVIEW_EVERY / PREVIEW_PAIRS | 1000 / 5 | Five fresh A/B/result JPEG triptychs, each using 30 sampling steps. |
| SAVE_EVERY / KEEP_LAST | 1000 / 3 | Save checkpoints on Colab SSD and upload in the background. Keep three checkpoints published by the uploader after Drive verifies them. |
| LEARNING_RATE / WARMUP_STEPS | 3e-5 / 1000 | AdamW8bit peak LR and initial warmup. |
| LR_CYCLES / MIN_LR_RATIO | 5 / 0.01 | Decaying cyclic cosine schedule. |

Per-block Inductor compilation is enabled. The CPU prefetch worker reads latent batches while the GPU runs. Tag shuffling and dropout remain dynamic. The launcher restores optimizer and scheduler state on subsequent resumes of this run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_NAME = DATASET_NAME + '-from-074000'
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 2
BUCKET_RESOLUTION = 512
PIXEL_BUDGET = BUCKET_RESOLUTION ** 2
SHUFFLE_BUFFER = 0  # Artist backend samples randomly from the full RAM index.
PREFETCH_BATCHES = 2  # Bounded CPU batch producer, overlapping data loading with GPU work.
RESUME_DATA_MODE = 'reseed'  # Draw a fresh dataset seed on each resume.
MAX_STEPS = 100_000
PREVIEW_EVERY, PREVIEW_PAIRS, PREVIEW_STEPS = 1000, 5, 30
SAVE_EVERY, KEEP_LAST = 1000, 3
LEARNING_RATE, WARMUP_STEPS = 3e-5, 1000
LR_CYCLES, MIN_LR_RATIO = 5, 0.01
COMPILE = True
CHECKPOINT_BLOCKS = 0  # 0 disables recomputation; 28 checkpoints every block.
TAG_DROPOUT = 0.10
CACHE_LATENTS = True
CACHE_LATENTS_TO_DISK = True  # Standard Anima NPZ cache on the Colab SSD.
VAE_BATCH_SIZE = 1
LATENT_CACHE_DIR = Path('/content/anima-latents')
LATENT_CACHE_RESERVE_RAM_GIB = 8  # Free RAM reserve beyond the bounded cache working batch.
OFFLOAD_TEXT_ENCODER = False  # Keep Qwen weights and compute on the GPU.
OUTPUT_DIR = DRIVE_ROOT / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config = json.loads((REPO / 'configs/anima_style_colab.json').read_text())
config['assets_manifest'] = str(MODELS / 'resolved_models.json')
config['reference_adapter'] = {'format': 'anima_checkpoint', 'path': str(INIT_CHECKPOINT)}
config['noise'] = {'distribution': 'progress_logit_normal', 'mean': -2.35, 'std': 1.05}
config['offload_text_encoder'] = OFFLOAD_TEXT_ENCODER
config['trim_text_padding'] = True
config['gradient_checkpointing'] = CHECKPOINT_BLOCKS > 0
config['checkpoint_blocks'] = CHECKPOINT_BLOCKS
config['tag_cache'] = str(TAG_CACHE)
config['content_loss'] = 'flow_l2'
config['cache_latents'] = CACHE_LATENTS
config['cache_latents_to_disk'] = CACHE_LATENTS_TO_DISK
config['vae_batch_size'] = VAE_BATCH_SIZE
config['latent_cache_dir'] = str(LATENT_CACHE_DIR)
config['latent_cache'] = {'reserve_ram_gib': LATENT_CACHE_RESERVE_RAM_GIB}
if USE_PRECOMPUTED_LATENTS:
    if not CACHE_LATENTS:
        raise ValueError('Enable CACHE_LATENTS to reuse the precomputed VAE cache.')
    config['latent_cache'].update(precomputed_dir=str(PRECOMPUTED_LATENT_DIR), load_to_memory=LATENTS_IN_MEMORY)
dataset = config['data']['datasets'][0]
dataset.update(batch_size=BATCH_SIZE, resolution=[BUCKET_RESOLUTION, BUCKET_RESOLUTION], max_pixels=PIXEL_BUDGET, shuffle_buffer=SHUFFLE_BUFFER, caption_tag_dropout_rate=TAG_DROPOUT)
config['data'].update(artist_index=str(INDEX_DIR), index_fetch_workers=2, index_cache_bytes=0)
config['training'].update(batch_size=BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION, max_steps=MAX_STEPS, save_every=SAVE_EVERY, keep_last_checkpoints=KEEP_LAST, learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS, lr_cycles=LR_CYCLES, min_lr_ratio=MIN_LR_RATIO, prefetch_batches=PREFETCH_BATCHES)
config['training'].update(resume_data_mode=RESUME_DATA_MODE, log_stage_timing=False)
config['sampling'].update(every_n_steps=PREVIEW_EVERY, num_pairs=PREVIEW_PAIRS, steps=PREVIEW_STEPS)
RUN_CONFIG = OUTPUT_DIR / 'launch_config.json'
def training_settings(value):
    settings = {key: item for key, item in value.items() if key not in ('cache_latents', 'cache_latents_to_disk', 'vae_batch_size', 'latent_cache_dir', 'latent_cache', 'checkpoint_cpu_offload', 'checkpoint_blocks', 'gradient_checkpointing', 'tag_cache', 'offload_text_encoder', 'trim_text_padding')}
    settings['training'] = {key: item for key, item in value['training'].items() if key != 'resume_data_mode'}
    return settings
if RUN_CONFIG.exists() and training_settings(json.loads(RUN_CONFIG.read_text())) != training_settings(config):
    raise RuntimeError('Existing run settings differ. Restore the settings or choose a new RUN_NAME.')
RUN_CONFIG.write_text(json.dumps(config, indent=2) + '\n')
print('Persistent run:', OUTPUT_DIR)
print(json.dumps(config['training'], indent=2))


Run the TensorBoard cell before launching training to keep its dashboard visible. Losses, learning rate and gradient norm are written to `metrics.jsonl`. Console output goes to `console.log`, and JPEG previews are under `samples/step-XXXXXX/` on Drive.
The launch cell requests Google Drive API access. A background worker uploads completed checkpoints from Colab SSD while training continues, verifies every file against Drive's size and checksum, then publishes the finished step folder. Drive stays mounted. Wait for `Drive verified step-XXXXXX` to know which step is recoverable from Drive. On normal exit the cell finishes queued uploads.

In [ ]:
TB_LOGDIR = str(OUTPUT_DIR / 'tensorboard')
%load_ext tensorboard
%tensorboard --logdir $TB_LOGDIR


In [ ]:
# Load the downloaded pretrained components and train the editing LoRA.
# Missing/incompatible checkpoints raise an error; there is no random fallback.
import shlex, sys
from concurrent.futures import ThreadPoolExecutor
from threading import Event
from google.colab import auth
sys.path.insert(0, str(REPO / 'tools/colab'))
from drive_checkpoints import DriveCheckpointPublisher, checkpoint_step, completed_checkpoints
AUTO_RESUME = True
auth.authenticate_user()
publisher = DriveCheckpointPublisher(OUTPUT_DIR, KEEP_LAST)
publisher.publish_pending()  # Retry completed SSD checkpoints from an earlier launch in this runtime.
checkpoints = sorted(completed_checkpoints(OUTPUT_DIR) + completed_checkpoints(publisher.staging_dir),
                     key=lambda path: checkpoint_step(path.name))
command = [str(VENV / 'bin/python'), '-u', str(REPO / 'anima_train_style.py'), '--config', str(RUN_CONFIG), '--output_dir', str(OUTPUT_DIR), '--device', 'cuda']
command += ['--checkpoint_dir', str(publisher.staging_dir)]
if CACHE_LATENTS:
    command += ['--cache_latents']
if CACHE_LATENTS_TO_DISK:
    command += ['--cache_latents_to_disk', '--latent_cache_dir', str(LATENT_CACHE_DIR)]
command += ['--vae_batch_size', str(VAE_BATCH_SIZE)]
if COMPILE:
    command += ['--compile', '--compile_backend', 'inductor', '--compile_scope', 'blocks', '--compile_modules', 'dit', '--compile_dynamic', 'true']
if AUTO_RESUME and checkpoints:
    command += ['--resume', str(checkpoints[-1])]
elif checkpoints:
    raise RuntimeError('Existing checkpoints found. Enable AUTO_RESUME or choose a new RUN_NAME.')
print(shlex.join(command), flush=True)
stop_uploads = Event()
with ThreadPoolExecutor(max_workers=1, thread_name_prefix='drive-checkpoint') as executor:
    uploads = executor.submit(publisher.watch, stop_uploads)
    try:
        with (OUTPUT_DIR / 'console.log').open('a', buffering=1) as log:
            log.write('\nCOMMAND: ' + shlex.join(command) + '\n')
            process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    if uploads.done():
                        uploads.result()
                if process.wait():
                    raise RuntimeError('Training failed. Read the output above and console.log on Drive.')
            finally:
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=20)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                process.stdout.close()
    finally:
        stop_uploads.set()
        uploads.result()  # Drain completed saves before this cell exits.
